# Course 6: Geometric Deep Learning and Graphs
## Session 11: Spatio-temporal graphs

**Goal:** forecast a signal on a graph of sensors and show what the graph adds over treating each sensor alone.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** every cell without a NOT RUN label was run once on a CPU (November 2026) on small built-in or synthetic data. Cells labelled NOT RUN need a library, a dataset download or a GPU; run them on Kaggle or Colab.

### 1. Simulated traffic on a road network
METR-LA needs a download, so the runnable part simulates 30 sensors. Congestion at one sensor spreads to its neighbours over the next steps, and there is a daily rhythm.

In [ ]:
import numpy as np
from sklearn.linear_model import Ridge
rng = np.random.default_rng(0)
n, T = 30, 3000
pos = rng.random((n, 2))
D = np.linalg.norm(pos[:, None] - pos[None], axis=-1)
A = ((D < np.sort(D, 1)[:, [4]]) & (D > 0)).astype(float); A = np.maximum(A, A.T)       # connect each sensor to its 3 nearest
P = A / A.sum(1, keepdims=True)

x = np.zeros((T, n)); shock = rng.normal(size=(T, n)) * (rng.random((T, n)) < 0.03) * 3
for t in range(1, T):
    x[t] = 0.5 * x[t - 1] + 0.45 * P @ x[t - 1] + shock[t] + 0.05 * rng.normal(size=n)
daily = np.sin(2 * np.pi * np.arange(T) / 288)[:, None]                                # 288 five-minute steps a day
speed = 60 - 8 * np.abs(x) + 5 * daily + rng.normal(size=(T, n)) * 0.5
print("sensors", n, "| time steps", T, "| mean speed", speed.mean().round(1))

### 2. Split by time, never at random
Train on the first 70%, validate on the next 10%, test on the last 20%. Scale with training statistics only.

In [ ]:
i_tr, i_va = int(0.7 * T), int(0.8 * T)
mu, sd = speed[:i_tr].mean(), speed[:i_tr].std()
z = (speed - mu) / sd
LAGS, H = 6, 3                                        # use the last 30 minutes to predict 15 minutes ahead

def windows(z, lo, hi, graph_hops):
    Xs, Ys = [], []
    for t in range(max(lo, LAGS), hi - H):
        past = z[t - LAGS:t]                          # (LAGS, n)
        feats = [past.T]                              # each sensor's own history
        M = past.T
        for _ in range(graph_hops):
            M = P @ M; feats.append(M)                # the average history of neighbours, then of their neighbours
        Xs.append(np.concatenate(feats, 1)); Ys.append(z[t + H - 1])
    return np.concatenate(Xs), np.concatenate(Ys)

### 3. Three forecasters
1. Persistence: the speed stays as it is now.
2. Own history only: a linear model per reading, with no graph.
3. Graph model: the same linear model, plus the neighbours' history (one and two hops). This is the diffusion-convolution idea with the nonlinearity and recurrence stripped away.

In [ ]:
def mae(pred, y): return float(np.abs(pred - y).mean() * sd)       # back in speed units

results = {}
Xte0, yte = windows(z, i_va, T, 0)
results["persistence"] = mae(Xte0[:, LAGS - 1], yte)
for name, hops in [("own history", 0), ("graph, 1 hop", 1), ("graph, 2 hops", 2)]:
    Xtr, ytr = windows(z, 0, i_tr, hops); Xte, _ = windows(z, i_va, T, hops)
    results[name] = mae(Ridge(alpha=1.0).fit(Xtr, ytr).predict(Xte), yte)
for k, v in results.items(): print(f"{k:14s} test MAE {v:.3f}")
assert results["graph, 1 hop"] < results["own history"] < results["persistence"]

### 4. A sanity check: shuffle the graph
If the gain really comes from the road structure, a random graph with the same number of edges should not help.

In [ ]:
perm = rng.permutation(n); P_true = P.copy()
P = P_true[perm][:, perm]                                 # wrong neighbours
Xtr, ytr = windows(z, 0, i_tr, 1); Xte, _ = windows(z, i_va, T, 1)
shuffled = mae(Ridge(alpha=1.0).fit(Xtr, ytr).predict(Xte), yte)
P = P_true
print(f"shuffled graph MAE {shuffled:.3f}  vs true graph {results['graph, 1 hop']:.3f}  vs no graph {results['own history']:.3f}")
assert shuffled > results["graph, 1 hop"]

The shuffled graph is clearly worse than the true one. It may still beat using no graph at all, because averaging any group of sensors picks up the shared daily rhythm. Keep that in mind when a paper reports a gain from 'the graph'.

### 5. METR-LA with a neural model (NOT RUN here)

In [ ]:
# NOT RUN: needs the METR-LA files (207 sensors, 5-minute readings, about 4 months) and a GPU for the neural model.
# The DCRNN repository links the data: metr-la.h5 and the sensor graph adj_mx.pkl.
import pandas as pd, pickle
df = pd.read_hdf("metr-la.h5")                                   # rows: time, columns: sensors
_, _, adj = pickle.load(open("adj_mx.pkl", "rb"), encoding="latin1")
# 1. Zeros in this file mean missing readings. Mask them in the loss and in the metric.
# 2. Split by time 70/10/20 and scale with training statistics.
# 3. Run the three forecasters above first. They are your baselines.
# 4. Then train a neural model (for example a GRU whose linear maps are replaced by graph convolutions) for horizons of 15, 30 and 60 minutes.
# 5. Report masked MAE at each horizon and compare with the DCRNN paper's table.

### Stretch (TODO): dengue across districts
Build a graph of Bangladesh's 64 districts (edges between districts that share a border), collect weekly case counts from DGHS bulletins, and forecast one and two weeks ahead. Use the same three baselines and the same time-based split. Report honestly if the graph does not help: reporting delays may matter more than geography.